# Procesamiento Final
Equipo 6

In [60]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

In [61]:
df = pd.read_csv('ConglomeradoTalux.csv', low_memory=False)
df.drop('Unnamed: 0', axis=1, inplace=True)

## Cliente/Leads

In [62]:
import csv
import random
from datetime import date, timedelta

inicio = date(1960, 1, 1)
fin = date(2000, 12, 31)
delta = fin - inicio

random.seed(33)
fechas = []
for i in range(1, 13935):
    dias_aleatorios = random.randint(0, delta.days)
    fecha = inicio + timedelta(days=dias_aleatorios)
    fecha_str = fecha.strftime("%d/%m/%Y")
    fechas.append(fecha_str)

In [63]:
# Insertamos las columnas de df que aplican y simulamos las nuevas.

clientes = df[['Cliente','Nombre cliente','E-mail','Telefono 1','Telefono 2','Localidad','C.P.','Colonia']].drop_duplicates(subset='Cliente').set_index('Cliente')

clientes['Nombre cliente'] = list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False).nombre)
clientes['Apellido'] = list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False, header=0).apellidos)
clientes['Genero'] = list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False, header=0).genero)
clientes['Fecha_de_nacimiento'] = fechas
clientes['id_empresa'] = [(i % 1000) + 1 for i in range(clientes.shape[0])]
clientes['id_ejecutivo'] = [(i % 15) + 1 for i in range(clientes.shape[0])]

clientes.reset_index(drop=False, inplace=True)

clientes.dropna(subset=['Cliente'], inplace=True)

clientes.rename(columns={'Cliente':'id_cliente'}, inplace=True)
clientes.rename(columns={'Nombre cliente':'Nombre'}, inplace=True)

#La columna es_lead toma el valor de verdadero cuando el id del cliente no aparece en la tabla ventas.
clientes['Es_lead'] = np.random.choice([True, False], size=len(clientes), p = [0.05,0.95])

In [64]:
# Simulamos datos que sean nulos y que no permitan nulls

clientes.loc[clientes['E-mail'].isna(), 'E-mail'] = (
    clientes.loc[clientes['E-mail'].isna(), 'Nombre'].str[:4].str.lower() + '_' +
    clientes.loc[clientes['E-mail'].isna(), 'Apellido'].str[:4].str.lower() + '@gmail.com'
)

In [65]:
# Reordenamos columnas para que siga el modelo de datos

clientes = clientes[['id_cliente', 'Nombre', 'Apellido', 'Genero', 'Fecha_de_nacimiento', 'E-mail', 'Telefono 1', 'Telefono 2', 'id_empresa', 'id_ejecutivo', 'Localidad', 'C.P.', 'Colonia', 'Es_lead']]

#Quitamos el único valor erróneo identificado
clientes = clientes[clientes['id_cliente'] != 'BI']
clientes

,id_cliente,Nombre,Apellido,Genero,Fecha_de_nacimiento,E-mail,Telefono 1,Telefono 2,id_empresa,id_ejecutivo,Localidad,C.P.,Colonia,Es_lead
0,166117,Alejandra,Peña Mendoza,F,01/08/1985,ggg2806831@gmail.com,8.129021e+09,8112856987,1,1,NUEVO LEON,64700,NaN,False
1,142823,Manuel Patricia,Ávila Miranda,M,03/07/1967,floresgarza.af@gmail.con,5.549131e+09,NaN,2,2,NUEVO LEON,64610,NaN,False
2,23105,Juan Ana,Ruiz Aguilar,M,11/05/1988,C.ADPERG@EXIROS.COM,8.119648e+09,NaN,3,3,NUEVO LEON,66450,NaN,False
4,69526,José,Cervantes Mendoza,M,24/08/1996,profra_armandinahdz@hotmail.com,8.118023e+09,NaN,5,5,NUEVO LEON,64700,NaN,False
5,19229,Luis,Delgado Ávila,M,18/09/1997,eitan.gara@fleetrenting.mx,8.110224e+09,8110223622,6,6,NUEVO LEON,64750,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13929,85870,Daniel,Díaz Molina,M,08/03/1989,azuchm@hotmail.com,8.110809e+09,8110808503,930,10,NUEVO LEON,64349,GRAN VIA CUMBRES,False
13930,73111,Juan Gabriela,Flores Ávila,M,02/02/1973,a_guillen_83@hotmail.com,8.183970e+09,8183969504,931,11,NUEVO LEON,64102,BARRIO ESTRELLA NORT,False
13931,53174,Leticia Beatriz,Miranda Cabrera,F,18/02/1970,drmunoz80@hotmail.com,NaN,NaN,932,12,NUEVO LEON,66350,ZONA INDUSTRIAL,True
13932,57558,Lucía,Rojas Espinoza,F,28/06/1990,lucí_roja@gmail.com,NaN,NaN,933,13,NUEVO LEON,66350,ZONA INDUSTRIAL,True


## Ventas

In [66]:
# Insertamos las columnas de df que aplican y simulamos las nuevas.

ventas = df[['T.factura','Cliente','VIN']].drop_duplicates(subset='T.factura').set_index('T.factura')
ventas.reset_index(drop=False, inplace=True)
ventas.dropna(inplace=True)

ventas['id_ejecutivo'] = [(i % 15) + 1 for i in range(ventas.shape[0])]

dates = pd.date_range(start='2017-01-01', end=pd.Timestamp.today(), freq='D')
sample = np.random.choice(dates.strftime('%d/%m/%Y'), size=1313, replace=True)
ventas['Fecha'] = sample

ventas['Duracion contrato'] = np.random.choice([24,36,48], size=1313, replace=True)
ventas['Contrato finalizado'] = pd.to_datetime(ventas['Fecha'], dayfirst = True) < datetime.today() - timedelta(days=4*365)

ventas.reset_index(drop=False, inplace=True)
ventas['index'] = ventas['index'].astype(int) + 1

In [67]:
ventas.rename(columns={'index': 'id_venta', 'T.factura': 'Monto', 'Cliente': 'id_cliente'}, inplace=True)

# Reordenamos columnas para que siga el modelo de datos

ventas = ventas[['id_venta', 'id_cliente', 'VIN', 'id_ejecutivo', 'Fecha', 'Monto',
                 'Duracion contrato', 'Contrato finalizado']]

ventas.loc[ventas['id_cliente'].isin(clientes[clientes['Es_lead'] == True]['id_cliente']), 'Monto'] = np.nan
ventas.loc[ventas['id_cliente'].isin(clientes[clientes['Es_lead'] == True]['id_cliente']), 'Contrato finalizado'] = False

ventas['Monto'] = np.abs(ventas['Monto'])

In [68]:
ventas

,id_venta,id_cliente,VIN,id_ejecutivo,Fecha,Monto,Duracion contrato,Contrato finalizado
0,1,166117,1HGCY1673SA900003,1,11/09/2020,765900.00,24,True
1,2,23105,1HGCY1677SA900067,2,16/08/2019,775900.00,36,True
2,5,19229,1HGCV1619LA900142,3,20/10/2017,400000.00,24,True
3,6,103957,1HGCV1615LA900090,4,02/07/2020,474900.00,48,True
4,7,128333,1HGCV1613LA900475,5,28/10/2024,480900.01,24,False
...,...,...,...,...,...,...,...,...
1308,2717,48718,5KBYF4885EB801813,4,23/02/2022,501064.99,48,True
1309,2718,97640,5KBYF6895GB801169,5,02/02/2024,580203.00,24,False
1310,2719,22480,5KBYF689XHB801122,6,15/11/2020,NaN,24,False
1311,2720,53174,5FPYK1654BB801368,7,28/02/2026,NaN,36,False


## Autos

In [69]:
# Insertamos las columnas de df que aplican y simulamos las nuevas.

autos = df[['VIN','Modelo','Color','Año Vehiculo']].drop_duplicates(subset='VIN').set_index('VIN')
autos.reset_index(drop=False, inplace=True)
autos = autos[autos['VIN'].isin(ventas['VIN'].unique())]

autos['Marca'] = list(pd.read_csv('Simulados/carros.csv', low_memory=False).Manufacturer)
autos['Año Vehiculo'] = list(pd.read_csv('Simulados/carros.csv', low_memory=False).Model_Year)
autos

,VIN,Modelo,Color,Año Vehiculo,Marca
0,1HGCY1673SA900003,ACCORD PRIME,NEGRO CRISTA,2025,Honda (USA)
2,1HGCY1677SA900067,ACCORD PRIME,BLANCO PLATI,2025,Honda (USA)
5,1HGCV1619LA900142,ACCORD EX CVT,BLANCO PLATI,2020,Honda (USA)
6,1HGCV1615LA900090,ACCORD EX CVT,PLATA LUNAR,2020,Honda (USA)
7,1HGCV1610LA900370,ACCORD EX CVT,PLATA LUNAR,2020,Honda (USA)
...,...,...,...,...,...
30372,5KBYF4885EB801813,Pilot Touring,PLATA DIAMAN,2014,Honda (Thailand)
30406,5KBYF6895GB801169,Pilot Touring,AZUL OBSIDIA,2016,Honda (Thailand)
30416,5KBYF689XHB801122,Pilot Touring,PLATA LUNAR,2017,Honda (Thailand)
30461,5FPYK1654BB801368,Ridgeline Rtl 2011,BLANCO MARFI,2011,Honda (USA)


## Ejecutivos

In [71]:
# Insertamos las columnas de df que aplican y simulamos las nuevas.

ejecutivos = pd.DataFrame(
    {'id_ejecutivo' : [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],
     'Nombre': list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False).nombre)[300:315],
     'Apellido': list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False).apellidos)[600:615],
     'Genero': list(pd.read_csv('Simulados/Clientes_nom_ape_gen.csv', low_memory=False).genero)[300:315],
     'Fecha_de_nacimiento' : fechas[600:615],
     'E-mail' : list(clientes['E-mail'][300:315]),
     'Telefono' : ['9191175013','9191094789','9191088722','9632587412','9615897548','9191875013','9191594789','9198788722','9632596412','9615637548','9191175363','9541094789','9871088722','9632512412','9615007548'],
     'Localidad' : list(clientes['Localidad'][900:915]),
     'C.P.' : clientes['C.P.'].unique()[-15:],
     'Colonia' : clientes['Colonia'].unique()[-15:],
     'Fecha_de_ingreso': ["01/01/2019", "01/07/2019", "02/01/2020", "08/07/2020", "04/09/2021", "01/07/2021", "11/11/2022", "12/11/2022", "04/04/2023", "09/09/2023", "30/05/2024", "03/09/2024", "05/11/2025", "07/12/2025", "11/10/2026"]})

ejecutivos

,id_ejecutivo,Nombre,Apellido,Genero,Fecha_de_nacimiento,E-mail,Telefono,Localidad,C.P.,Colonia,Fecha_de_ingreso
0,1,Miguel Cristina,Jiménez Herrera,M,18/11/1983,oziel_ajt1@hotmail.com,9191175013,NUEVO LEON,72410,LORENZO GARZA,01/01/2019
1,2,Patricia,Morales Lara,F,30/10/1999,ana.cues.davila@hotmail.com,9191094789,NUEVO LEON,36251,CAMPOBELLO II,01/07/2019
2,3,Mariana,Torres Herrera,F,23/12/1968,galaviz.jorge@hotmail.com,9191088722,NUEVO LEON,26015,FRACC. LA DIANA,02/01/2020
3,4,Miguel Norma,Estrada Díaz,M,20/05/1990,arq.marisolcc@hotmail.com,9632587412,NUEVO LEON,38080,BARRIO ESTRELLA NORT,08/07/2020
4,5,Eduardo Alejandra,Morales Acosta,M,03/09/1988,gg104055@gmail.com,9615897548,NUEVO LEON,87028,COLIMA CENTRO,04/09/2021
5,6,Enrique Arturo,Ramírez Cabrera,M,23/12/1981,sofiaywenn@hotmail.com,9191875013,NUEVO LEON,66025,CIUDAD GRANJA,01/07/2021
6,7,Ramón Héctor,Ochoa Ruiz,M,29/01/1994,cgoc2003@yahoo.com.mx,9191594789,NUEVO LEON,64442,FRACC ALMAGUER,11/11/2022
7,8,Alejandro,Flores Núñez,M,17/07/1965,jeronimomtzjr@gmail.com,9198788722,NUEVO LEON,31125,SARABIA,12/11/2022
8,9,Laura Rafael,Lara Espinoza,F,10/07/1960,katia.sofia94@hotmail.com,9632596412,COAHUILA,64319,FRACC. CUMBRES DE SA,04/04/2023
9,10,Rosa Javier,Estrada Martínez,F,26/06/1987,ramó_pach@gmail.com,9615637548,NUEVO LEON,27020,FRACC AMISTAD,09/09/2023


## Empresa

In [72]:
empresas = pd.DataFrame({
    "id_empresa" : list(range(1,1001)),
    "Nombre" : list(pd.read_csv('Simulados/Empresas.csv', low_memory=False).Empresa),
    "Giro" : np.random.choice(['Manufactura', 'Servicios', 'Comercial'], p=[1/3,1/3,1/3], size = 1000),
    "Tamaño personal" : np.random.randint(10,3000, size = 1000),
    "Localidad" : list(clientes['Localidad'][10000:11000]),
    "C.P." : list(clientes['C.P.'][10000:11000]),
    "Colonia" : list(clientes['Colonia'][10000:11000]),
})

empresas

,id_empresa,Nombre,Giro,Tamaño personal,Localidad,C.P.,Colonia
0,1,Inversiones Morelos y Cía.,Servicios,1940,NUEVO LEON,67150,FRACC. LA FUENTE
1,2,Tecnología Regional del Sur y Cía.,Manufactura,2935,NUEVO LEON,66417,RDCIAL. BORANA
2,3,Inmobiliaria Regional Anáhuac S.A.,Servicios,1270,NUEVO LEON,67484,JOSE O MARTINEZ
3,4,Inversiones Michoacán y Cía.,Manufactura,2568,NUEVO LEON,64890,VILLA LAS FUENTES
4,5,Industrias Zacatecas S.A.,Servicios,1597,NUEVO LEON,64920,LOMAS DEL PASEO RESI
...,...,...,...,...,...,...,...
995,996,Inmobiliaria Jalisco San Miguel S.C.,Servicios,960,NUEVO LEON,67118,FRACC VILLA ESPAÑOLA
996,997,Consultora Jalisco Internacional,Manufactura,2049,NUEVO LEON,64610,LAS CUMBRES 4TO SECT
997,998,Grupo Colima Nayarit Internacional,Manufactura,656,NUEVO LEON,64610,CUMBRES 5TO SECTOR
998,999,Grupo Allende Hermanos,Manufactura,1915,NUEVO LEON,64344,CUMBRES SANTA CLARA


## Arreglando Problemitas Finales

In [73]:
clientes['Fecha_de_nacimiento'] = pd.to_datetime(clientes['Fecha_de_nacimiento'], format = "%d/%m/%Y")
clientes['Telefono 1'] = pd.to_numeric(clientes['Telefono 1'], errors="coerce").astype('float64').astype('Int64')
clientes['Telefono 2'] = pd.to_numeric(clientes['Telefono 2'], errors="coerce")
clientes['Telefono 2'] = clientes['Telefono 2'].apply(lambda x: int(float(x)) if pd.notna(x) else None).astype('Int64')
clientes['C.P.'] = pd.to_numeric(clientes['C.P.'], errors="coerce").astype('float64').astype('Int64')

ventas['Fecha'] = pd.to_datetime(ventas['Fecha'], format = "%d/%m/%Y")

autos['Año Vehiculo'] = pd.to_numeric(autos['Año Vehiculo'], errors="coerce").astype('float64').astype('Int64')

ejecutivos['Telefono'] = pd.to_numeric(ejecutivos['Telefono'], errors="coerce").astype('float64').astype('Int64')
ejecutivos['C.P.'] = pd.to_numeric(ejecutivos['C.P.'], errors="coerce").astype('float64').astype('Int64')
ejecutivos['Fecha_de_nacimiento'] = pd.to_datetime(ejecutivos['Fecha_de_nacimiento'], format = "%d/%m/%Y")
ejecutivos['Fecha_de_ingreso'] = pd.to_datetime(ejecutivos['Fecha_de_ingreso'], format = "%d/%m/%Y")

empresas['C.P.'] = pd.to_numeric(empresas['C.P.'], errors="coerce").astype('float64').astype('Int64')

## Exportar Tablas

In [74]:
clientes.to_csv('TablasFinales/clientes_leads.csv', index=False)
autos.to_csv('TablasFinales/autos.csv', index=False)
ejecutivos.to_csv('TablasFinales/ejecutivos.csv', index=False)
ventas.to_csv('TablasFinales/ventas.csv', index=False)
empresas.to_csv('TablasFinales/empresas.csv', index=False)